# 工具的使用

In [1]:
# 无参数类装饰器	类__init__接收函数作为参数，__call__接受参数
import functools

class CountCalls:
    """统计函数调用次数的装饰器"""
    def __init__(self, func):
        # functools.wraps(func)(self)  # 保留原函数元数据
        self.func = func
        self.count = 0
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"调用 {self.func.__name__} 第 {self.count} 次")
        return self.func(*args, **kwargs)

# 使用
@CountCalls
def say_hello(name):
    print("你好", name)
    return "最后返回结果"

print(say_hello("靓仔"))
print(say_hello("杭州"))
print(f"总调用次数: {say_hello.count}")

调用 say_hello 第 1 次
你好 靓仔
最后返回结果
调用 say_hello 第 2 次
你好 杭州
最后返回结果
总调用次数: 2


In [2]:
# 带参数的类装饰器通过 __init__ 接收参数，__call__ 接收被装饰的函数，形成两层调用结构
class DecoratorWithArgs:
    def __init__(self, arg1, arg2):      # 接收装饰器参数
        self.arg1 = arg1
        self.arg2 = arg2
    
    def __call__(self, func):             # 接收被装饰的函数
        def wrapper(*args, **kwargs):
            # 使用 self.arg1, self.arg2
            print("装饰器参数：", self.arg1, self.arg2)
            return func(*args, **kwargs)
        return wrapper

# 使用方式
@DecoratorWithArgs(arg1=10, arg2="参数2")
def say_hello(name):
    print("你好", name)
    return "最后返回结果"

print(say_hello("靓仔"))

装饰器参数： 10 参数2
你好 靓仔
最后返回结果


In [1]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.runnables import RunnableConfig

In [6]:
@tool
def get_message(username: str):
    """
    获取信息
    """
    # print(runtime.state)
    return f"1{username}"

In [7]:
agent = create_agent(
    model=init_chat_model(
        model="ollama:qwen2.5:7b-instruct-q4_K_M"
    ),
    tools=[get_message],
    system_prompt="你是一个数据助手，专门处理用户数据。用户询问数据量时，调用`get_message`方法返回数据"
)

agent.invoke(
    input={
        "messages": [
            {"role": "user", "content": "当前'wjc'用户访问的数据量？"}
        ]
    }
)

{'messages': [HumanMessage(content="当前'wjc'用户访问的数据量？", additional_kwargs={}, response_metadata={}, id='c9e3fb62-fcf4-4c8e-a82f-1505e357fd03'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b-instruct-q4_K_M', 'created_at': '2026-08-27T03:52:30.4423099Z', 'done': True, 'done_reason': 'stop', 'total_duration': 880672700, 'load_duration': 3192600, 'prompt_eval_count': 160, 'prompt_eval_duration': 260344000, 'eval_count': 21, 'eval_duration': 612829000, 'logprobs': None, 'model_name': 'qwen2.5:7b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--01a04159-0a78-7a91-a86b-d508fbb44161-0', tool_calls=[{'name': 'get_message', 'args': {'username': 'wjc'}, 'id': '518d067b-44dc-49df-987a-f23f880e3035', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 160, 'output_tokens': 21, 'total_tokens': 181}),
  ToolMessage(content='1wjc', name='get_message', id='73207276-c063-40cd-bf12-78fe6ff8f227', tool_call_id='518d067b-44dc-49df-